In [0]:
from pyspark.sql.types import StringType, IntegerType, DateType, BooleanType, TimestampType
import pyspark.sql.functions as F
from delta.tables import DeltaTable

In [0]:
dbutils.widgets.text("catalog_name", "ecommerce", "Catalog Name")
dbutils.widgets.text("storage_account_name", "stgecommerceeastus001", "Storage Account Name")
dbutils.widgets.text("container_name", "ecomm-raw-data", "Container Name")

In [0]:
catalog_name = dbutils.widgets.get("catalog_name")
storage_account_name = dbutils.widgets.get("storage_account_name")
container_name = dbutils.widgets.get("container_name")

In [0]:
silver_checkpoint_path = f"abfss://{container_name}@{storage_account_name}.dfs.core.windows.net/checkpoint/silver/fact_order_returns/"

In [0]:
df = spark.readStream \
.format("delta") \
.table(f"{catalog_name}.bronze.brz_order_returns")

In [0]:
df = df.dropDuplicates(['order_id', 'order_dt', 'return_ts'])

In [0]:
df = (
    df
    .withColumn("order_dt", F.col("order_dt").cast(DateType()))
    .withColumn("return_ts", F.col("return_ts").cast(TimestampType()))
    .withColumn("reason", F.upper(F.trim(F.col("reason"))))
    .withColumn("processed_time", F.current_timestamp())
    .withColumn(
        "primary_key",
        F.sha2(
            F.concat_ws(
                "||",
                F.coalesce(F.col("order_id").cast("string"), F.lit("NA")),
                F.coalesce(
                    F.date_format(F.col("return_ts"), "yyyy-MM-dd HH:mm:ss"),
                    F.lit("NA")
                )
            ),
            256
        )
    )
)

In [0]:


def upsert_to_silver(microBatchDF, batch_id):
    table_name = f"{catalog_name}.silver.slv_order_returns"

    # 🔹 Deduplicate incoming batch (critical for merge correctness)
    microBatchDF = microBatchDF.dropDuplicates(["primary_key"])

    if not spark.catalog.tableExists(table_name):
        # 🔹 First time: create table
        (
            microBatchDF.write
            .format("delta")
            .mode("overwrite")
            .saveAsTable(table_name)
        )

        # Enable Change Data Feed (CDF)
        spark.sql(f"""
            ALTER TABLE {table_name} SET TBLPROPERTIES (
                delta.enableChangeDataFeed = true
            )
        """)

    else:
        # Load existing Delta table
        delta_table = DeltaTable.forName(spark, table_name)

        # Perform UPSERT (MERGE)
        (
            delta_table.alias("silver_table")
            .merge(
                microBatchDF.alias("batch_table"),
                "silver_table.primary_key = batch_table.primary_key"
            )
            .whenMatchedUpdateAll()
            .whenNotMatchedInsertAll()
            .execute()
        )

In [0]:
df.writeStream.trigger(availableNow=True) \
    .foreachBatch(upsert_to_silver) \
    .option("checkpointLocation", silver_checkpoint_path) \
    .option('mergeSchema', 'true') \
    .outputMode('update') \
    .start() \
    .awaitTermination()